# 09 — Global admin0 children exposure (counts only)

One row per country, one sheet: **child population** (Total / Boys / Girls) followed by a column group per hazard with the number of **exposed children** (Total / Boys / Girls). No hazard levels, indices or classes.

## Inputs

- Global per-hazard admin2 CSVs from notebook 02: `{SOURCE_FOLDER}/{hazard}_{date}.csv`
- Independent admin2 population: `{POP_REFERENCE}` (same denominator as notebook 07)

## Outputs

- `{OUTPUT_FOLDER}/global_admin0/children_exposure_{date}.xlsx` — the grouped single-sheet view
- `{OUTPUT_FOLDER}/global_admin0/CSV/children_exposure_long_{date}.csv` — country × hazard, numeric, with `status`

## Model (same as notebook 07's admin0 counts)

- Population from the reference, so it is identical for every hazard; summed admin2 → admin0.
- Exposure summed admin2 → admin0 via `agg.aggregate_counts`. `No Data` units (raster does not cover them) count as **0 exposed**.
- `Insufficient Data` / `Not Applicable` — taken from the source file's own `status` — are always country-wide: the hazard cells are left **blank** and the status label is written in the Excel instead of a misleading 0.
- `agg.POP_BLANK_COUNTRIES` get no row at all.

In [ ]:
import sys
from pathlib import Path

SUBNATIONAL_ROOT = Path.cwd().parent
LIB_DIR = SUBNATIONAL_ROOT / "lib"
CONFIG_DIR = SUBNATIONAL_ROOT / "config"
for p in (str(SUBNATIONAL_ROOT), str(LIB_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import os
import pandas as pd
import aggregation_functions as agg
from pipeline import excel_report as xr

pd.set_option("display.max_columns", 500)

In [ ]:
# ============================================================
# Parameters
# ============================================================
PROD_DATE = "2026-08-31"
SOURCE_FOLDER = r"C:\Users\anjpu\Desktop\UNICEF\repo\CCRI\new_scripts\output_mali\global\CSV"
OUTPUT_FOLDER = "output"
POP_REFERENCE = f"{OUTPUT_FOLDER}/aux/admin2_population.csv"

# Children only: (label, exposure column, population column)
CHILD = {"total": ("u18_aexp", "u18_pop_total"),
         "boys":  ("u18_amexp", "u18_pop_m"),
         "girls": ("u18_afexp", "u18_pop_f")}
SUPPRESSED = list(agg.EXCLUDED_STATUSES)  # Insufficient Data / Not Applicable

os.makedirs(f"{OUTPUT_FOLDER}/global_admin0/CSV", exist_ok=True)
pop_ref = pd.read_csv(POP_REFERENCE)  # required: no fallback, the denominator must not drift

cfg = xr.load_report_config(CONFIG_DIR)
hazard_info = pd.read_json(CONFIG_DIR / "hazard_info.json")
discovered = agg.discover_hazard_codes(SOURCE_FOLDER, PROD_DATE)
codes = [c for c in hazard_info["code"] if c in discovered]  # hazard_info order
dropped = [c for c in discovered if c not in codes]
if dropped:
    print(f"skipping {len(dropped)} hazard(s) not in hazard_info.json: {dropped}")
print(f"hazards: {len(codes)} | population reference: {len(pop_ref):,} admin2")

In [ ]:
# ============================================================
# Per hazard: admin2 -> admin0 child exposure counts (long form)
# ============================================================
META = ["iso3", "adm0_ucode", "adm0_name", "unicef_region"]
frames = []
for code in codes:
    df = agg.load_hazard_global(code, PROD_DATE, SOURCE_FOLDER)
    df = agg.apply_population_reference(df, pop_ref)
    df = df[~df["adm0_ucode"].isin(agg.POP_BLANK_COUNTRIES)]

    # Country-level status from the SOURCE file. Suppression is country-wide in
    # the production data; guard it, because a partial flag would make the
    # blank-the-country rule below wrong.
    sup = df["status"].isin(SUPPRESSED).groupby(df["adm0_ucode"])
    frac = sup.mean()
    partial = frac[(frac > 0) & (frac < 1)]
    assert partial.empty, f"[{code}] partially suppressed countries: {list(partial.index)}"
    country_status = (df[df["status"].isin(SUPPRESSED)]
                      .groupby("adm0_ucode")["status"].first())

    a0 = agg.aggregate_counts(df, "adm0_ucode")
    a0["hazard_code"] = code
    a0["status"] = a0["adm0_ucode"].map(country_status).fillna("")
    # aggregate_counts treats null exposure as 0 (right for No Data); for a
    # suppressed country there is no result at all, so blank it.
    flagged = a0["status"] != ""
    for exp_col, _ in CHILD.values():
        a0.loc[flagged, exp_col] = pd.NA
    frames.append(a0[META + ["hazard_code"] + [c for pair in CHILD.values() for c in pair] + ["status"]])
    print(f"  {code}: {len(a0)} countries, {flagged.sum()} flagged")

long = pd.concat(frames, ignore_index=True)
long_path = f"{OUTPUT_FOLDER}/global_admin0/CSV/children_exposure_long_{PROD_DATE}.csv"
long.to_csv(long_path, index=False)
print(f"\nlong: {long.shape} -> {long_path}")

In [ ]:
# ============================================================
# Wide single-sheet Excel: Child Population group + one group per hazard
# ============================================================
POP_GROUP = "child_population"  # pseudo-code: gets a grouped header like a hazard
LABEL = {"total": "Total", "boys": "Boys", "girls": "Girls"}

# Population is hazard-independent (checked below): take it from the first hazard.
pop = (long[long["hazard_code"] == codes[0]]
       .set_index("adm0_ucode")[META[:1] + META[2:] + [p for _, p in CHILD.values()]])
wide = pop.rename(columns={p: f"{POP_GROUP}__{k}" for k, (_, p) in CHILD.items()})

for code in codes:
    h = long[long["hazard_code"] == code].set_index("adm0_ucode")
    for k, (e, _) in CHILD.items():
        # Flagged countries show their status text instead of an empty cell.
        wide[f"{code}__{k}"] = h[e].astype(object).where(h["status"] == "", h["status"])

wide = wide.reset_index().sort_values("adm0_ucode")
spans = [(POP_GROUP, [f"{POP_GROUP}__{k}" for k in CHILD])] + \
        [(c, [f"{c}__{k}" for k in CHILD]) for c in codes]

cfg_x = {**cfg, "code_display": {**cfg["code_display"], POP_GROUP: "Child Population (under 18)"}}
FRONT = ["iso3", "adm0_ucode", "adm0_name", "unicef_region"]
FRONT_LABELS = {"iso3": "ISO3", "adm0_ucode": "Country Code", "adm0_name": "Country",
                "unicef_region": "UNICEF Region"}

xlsx_path = f"{OUTPUT_FOLDER}/global_admin0/children_exposure_{PROD_DATE}.xlsx"
res = xr.write_wide_excel(
    wide, xlsx_path, cfg=cfg_x, front_cols=FRONT, group_spans=spans,
    class_value_suffixes=set(), front_labels=FRONT_LABELS, variable_labels=LABEL,
    thousands_cols={c for _, cols in spans for c in cols}, sheet_name="children_exposure",
)
print(res)

In [ ]:
# ============================================================
# Verification
# ============================================================
pop_cols = [p for _, p in CHILD.values()]
exp_cols = [e for e, _ in CHILD.values()]

# 1. Population identical for every hazard (the denominator must not drift).
per_hazard = long.groupby("hazard_code")[pop_cols].sum()
print("population identical across hazards:", (per_hazard.nunique() == 1).all())

# 2. Population equals the reference (minus the countries with no summary).
geo = agg.load_hazard_global(codes[0], PROD_DATE, SOURCE_FOLDER)[["adm2_ucode", "adm0_ucode"]]
ref = pop_ref.merge(geo, on="adm2_ucode")
ref = ref[~ref["adm0_ucode"].isin(agg.POP_BLANK_COUNTRIES)]
print(f"child population vs reference: {per_hazard['u18_pop_total'].iloc[0]:,.0f} "
      f"vs {ref['pop_under_18_total'].sum():,.0f}")

# 3. Boys + girls vs total. Not exact: the admin2 counts are rounded per sex in
#    the source, so small exposures drift by a few children per admin2 unit.
#    Relative error is only meaningful where the total is not tiny.
for what, t, b, g in [("population", "u18_pop_total", "u18_pop_m", "u18_pop_f"),
                      ("exposed", "u18_aexp", "u18_amexp", "u18_afexp")]:
    d = (long[b] + long[g] - long[t]).abs()
    rel = d / long[t].where(long[t] >= 10_000)
    print(f"{what}: max |boys+girls-total| = {d.max():,.0f} | "
          f"max relative (total >= 10k) = {rel.max():.3%}")

# 4. Exposed never exceeds population.
over = long[(long["u18_aexp"] > long["u18_pop_total"] + 1)]
print("rows with exposed > population:", len(over))

# 5. Conservation: admin0 exposed == admin2 source sum for unflagged countries.
for code in codes[:3]:
    src = agg.load_hazard_global(code, PROD_DATE, SOURCE_FOLDER)
    src = src[~src["adm0_ucode"].isin(agg.POP_BLANK_COUNTRIES) & ~src["status"].isin(SUPPRESSED)]
    got = long.loc[long["hazard_code"] == code, "u18_aexp"].sum()
    print(f"  [{code}] admin2={src['u18_aexp'].sum():,.0f} admin0={got:,.0f}")

print("\nstatus counts (country x hazard):")
print(long["status"].replace("", "(ok)").value_counts())